In [ ]:
#CELL: [1]

import os
import sys

import pandas as pd
from dateutil import parser

from SRC.CORE._CONSTANTS import _BINANCE_FEE
from SRC.EVALUATION.evaluate_model import prepare_data, fetch_data
from SRC.LIBRARIES.new_data_utils import get_group

os.environ[_BINANCE_FEE] = '0.0 - 0.0'

try:
    symbol = symbol
except:
    symbol = 'ZECUSDT'

try:
    start_dt_str = start_dt_str
except:
    start_dt_str = '2026-01-01'


start_dt = parser.parse(start_dt_str)

market_type = 'FUTURES'

fetched_data = fetch_data(market_type, symbol, start_dt_str)
df_origin_s = fetched_data['df_origin_s']
group_df = fetched_data['group_df']
historical_df = fetched_data['historical_df']

#CELL: [1]

In [ ]:
#CELL: [2]

prepared_data = prepare_data(model_name, df_origin_s, group_df, start_dt_str)
df_s = prepared_data['df_s']
group_constraints_df = prepared_data['group_constraints_df']
model = prepared_data['model']
segments = prepared_data['segments']
threshold = prepared_data['threshold']

# signal1 = model.predict_signal(group)
# signal2 = model.predict_signal(last_group)
# print(signal1)
# print(signal2)

#CELL: [2]

In [ ]:
#CELL: [3]

from SRC.CORE._CONSTANTS import _KIEV_TIMESTAMP

_actual_trp = f'tpr_{threshold}'
_predicted_trp = f'pred_tpr_{threshold}'

first_group = get_group(group_constraints_df.iloc[0], segments, df_s)
last_group = get_group(group_constraints_df.iloc[-1], segments, df_s)

result_df = pd.DataFrame(columns=[*first_group[0].columns.to_list(), _predicted_trp])

counter = 0
for idx, row in group_constraints_df.iterrows():
    group = get_group(row, segments, df_s)
    out_df = group[-1]
    out_row = out_df.iloc[0]

    signal = model.predict_signal(group)

    out_row[_predicted_trp] = signal['pred_tpr']
    result_df.loc[len(result_df)] = out_row

    if counter % 500 == 0:
        print(f'Processed {counter} groups: {out_row[_KIEV_TIMESTAMP]}, actual TPR: {out_row[_actual_trp]}, predicted TPR: {out_row[_predicted_trp]}')

    counter += 1

print(f'Processed {counter} groups: {out_row[_KIEV_TIMESTAMP]}, actual TPR: {out_row[_actual_trp]}, predicted TPR: {out_row[_predicted_trp]}')

#CELL: [3]

In [ ]:
#CELL: [4]

%load_ext autoreload
%autoreload

import numpy as np
from SRC.LIBRARIES.new_data_utils import validate_timeseries_df

result_df = result_df.dropna(subset=[_predicted_trp, _actual_trp])
result_df["tpr_sq_errors"] = (result_df[_predicted_trp] - result_df[_actual_trp])**2

mse = np.mean(result_df["tpr_sq_errors"].to_numpy())
print(f'Mean Squared Error: {mse}')

validate_timeseries_df(result_df, feature=_KIEV_TIMESTAMP)

#CELL: [4]

In [ ]:
from IPython.core.display import HTML
from SRC.LIBRARIES.new_utils import check_env_true
#CELL: [5]

from SRC.CORE._CONSTANTS import project_root_dir
from plotly.subplots import make_subplots
from SRC.LIBRARIES.new_plot_utils import plot_candles_zigzags, plot_volumes, plot_continous_tpr_pred_true_bars, plot_feature_line

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.45, 0.15, 0.25, 0.15],
    subplot_titles=[
        "Candlestick",
        "Volume",
        "Actual vs Predicted TPR",
        "Squared Error"
    ]
)

plot_candles_zigzags(df=result_df, fig=fig, fig_row=1, time_feature=_KIEV_TIMESTAMP, threshold=threshold)
plot_volumes(df=result_df, fig=fig, fig_row=2, time_feature=_KIEV_TIMESTAMP)
plot_continous_tpr_pred_true_bars(df=result_df, fig=fig, fig_row=3, time_feature=_KIEV_TIMESTAMP, threshold=threshold)
plot_feature_line(df=result_df, fig=fig, fig_row=4, time_feature=_KIEV_TIMESTAMP, feature='tpr_sq_errors', color='yellow')

fig.update_layout(
    title=f"Price, Volume, TPR Prediction and Error | Model: {model_name} | {symbol} | MSE: {mse:.7f}",
    barmode="group",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0
    )
)

fig.update_layout(
    height=900,
    xaxis_rangeslider_visible=False
)

html_file_path = f"{project_root_dir()}/OUT/ANALYZE/evaluate__{model_name}_{symbol}.html"
fig.write_html(html_file_path, auto_open=True)

if check_env_true('PRINT_OUT', False):
    display(HTML(filename=html_file_path))

#CELL: [5]